# Daniel Graham Model Testing
## xboost & KNN
Dataset
https://archive.ics.uci.edu/dataset/2/adult 

Environment based on Anaconda (python >= 3.13 and having install xgboost `conda install xgboost`)

### Imports and Initial Loading of Dataframe and Pipelines

In [24]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, f1_score, roc_auc_score, confusion_matrix
from sklearn.impute import KNNImputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier # type: ignore


In [25]:
all_columns = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 
    'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 
    'capital-loss', 'hours-per-week', 'native-country', 'income']
categorical_features = ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']
numerical_features = ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
target_feature = 'income'
random_state_value = 48

df = pd.read_csv('adult.data.csv', header=None, names=all_columns)
df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [26]:
X = df[categorical_features + numerical_features]
y = df[target_feature].apply(lambda x: 1 if x == ' >50K' else 0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_state_value, stratify=y)

categorical_transformer = Pipeline(steps=[
    # ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
numerical_transformer = Pipeline(steps=[
    # ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

### XGBoost (no hyperparameter tuning)

In [27]:
xgb = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42)

xgb_model = Pipeline(steps=[("preprocessor", preprocessor), ("classifier", xgb)])
xgb_model.fit(X_train, y_train)
predictions = xgb_model.predict(X_test)
predictions_proba = xgb_model.predict_proba(X_test)[:, 1]
confusion = confusion_matrix(y_test, predictions)
print("Confusion Matrix:")
print(confusion)
print("\nClassification Report:")
print(classification_report(y_test, predictions))
print("\nPredicted Probabilities:")
print(predictions_proba)
print("Accuracy:", np.mean(predictions == y_test.values))
print("Precision:", confusion[1, 1] / (confusion[0, 1] + confusion[1, 1]))
print("Recall:", confusion[1, 1] / (confusion[1, 0] + confusion[1, 1]))
# F1 score
xgb_f1 = f1_score(y_test, predictions)
print("F1 Score:", xgb_f1)
# ROC AUC
xgb_auc = roc_auc_score(y_test, predictions_proba)
print("XGB AUC:", xgb_auc)
# Cross-validation accuracy (5-fold)
xgb_cv_scores = cross_val_score(xgb_model, X, y, cv=5, scoring='accuracy')
print("XGB CV Accuracy scores:", xgb_cv_scores)
print("XGB CV Accuracy (mean):", np.mean(xgb_cv_scores))

Confusion Matrix:
[[4641  304]
 [ 506 1062]]

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.94      0.92      4945
           1       0.78      0.68      0.72      1568

    accuracy                           0.88      6513
   macro avg       0.84      0.81      0.82      6513
weighted avg       0.87      0.88      0.87      6513


Predicted Probabilities:
[0.0080122  0.00662933 0.01188828 ... 0.01720138 0.7144648  0.04123841]
Accuracy: 0.8756333486872409
Precision: 0.7774524158125915
Recall: 0.6772959183673469
F1 Score: 0.7239263803680982
XGB AUC: 0.9305912873238275
XGB CV Accuracy scores: [0.86933825 0.87039312 0.87576781 0.87822482 0.87684275]
XGB CV Accuracy (mean): 0.8741133495624513


### KNN (no hyperparameter tuning)

In [28]:
# KNN model using the same preprocessing pipeline as the XGBoost model
knn = KNeighborsClassifier(n_neighbors=5, weights='uniform')
knn_model = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', knn)])
knn_model.fit(X_train, y_train)
knn_predictions = knn_model.predict(X_test)
knn_probabilities = knn_model.predict_proba(X_test)[:, 1]

knn_confusion = confusion_matrix(y_test, knn_predictions)
print('KNN Confusion Matrix:')
print(knn_confusion)
print('\nKNN Classification Report:')
print(classification_report(y_test, knn_predictions))
print('\nKNN Predicted Probabilities:')
print(knn_probabilities)
print('KNN Accuracy:', np.mean(knn_predictions == y_test.values))
print('KNN Precision:', knn_confusion[1, 1] / (knn_confusion[0, 1] + knn_confusion[1, 1]))
print('KNN Recall:', knn_confusion[1, 1] / (knn_confusion[1, 0] + knn_confusion[1, 1]))
knn_f1 = f1_score(y_test, knn_predictions)
print('KNN F1 Score:', knn_f1)
# ROC AUC
knn_auc = roc_auc_score(y_test, knn_probabilities)
print('KNN AUC:', knn_auc)
# Cross-validation accuracy (5-fold)
knn_cv_scores = cross_val_score(knn_model, X, y, cv=5, scoring='accuracy')
print('KNN CV Accuracy scores:', knn_cv_scores)
print('KNN CV Accuracy (mean):', np.mean(knn_cv_scores))


KNN Confusion Matrix:
[[4470  475]
 [ 603  965]]

KNN Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.90      0.89      4945
           1       0.67      0.62      0.64      1568

    accuracy                           0.83      6513
   macro avg       0.78      0.76      0.77      6513
weighted avg       0.83      0.83      0.83      6513


KNN Predicted Probabilities:
[0.  0.  0.  ... 0.  0.4 0.2]
KNN Accuracy: 0.8344848764010441
KNN Precision: 0.6701388888888888
KNN Recall: 0.6154336734693877
KNN F1 Score: 0.6416223404255319
KNN AUC: 0.8681845324075029
KNN CV Accuracy scores: [0.82588669 0.82724201 0.83261671 0.83952703 0.83614865]
KNN CV Accuracy (mean): 0.832284217239307


### XGBoost (hyperparameter tuning)

In [29]:
# Hyperparameter tuning for XGBoost using GridSearchCV
xgb_param_grid = {
    'classifier__n_estimators': [100, 300, 500],
    'classifier__max_depth': [3, 5, 7],
    'classifier__learning_rate': [0.01, 0.05, 0.1],
    'classifier__subsample': [0.6, 0.8, 1.0],
    'classifier__colsample_bytree': [0.6, 0.8, 1.0]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state_value)

xgb_grid = GridSearchCV(xgb_model, param_grid=xgb_param_grid, cv=cv, scoring='accuracy', n_jobs=-1, verbose=1)

xgb_grid.fit(X_train, y_train)
print('XGB Best Params:', xgb_grid.best_params_)
print('XGB Best CV Accuracy:', xgb_grid.best_score_)

# Evaluate best estimator on test set
xgb_best = xgb_grid.best_estimator_
xgb_preds = xgb_best.predict(X_test)
xgb_proba = xgb_best.predict_proba(X_test)[:, 1]

xgb_conf = confusion_matrix(y_test, xgb_preds)
print('\nXGB Confusion Matrix:')
print(xgb_conf)
print('\nXGB Classification Report:')
print(classification_report(y_test, xgb_preds))
print('XGB Accuracy:', np.mean(xgb_preds == y_test.values))
print('XGB Precision:', xgb_conf[1, 1] / (xgb_conf[0, 1] + xgb_conf[1, 1]))
print('XGB Recall:', xgb_conf[1, 1] / (xgb_conf[1, 0] + xgb_conf[1, 1]))
print('XGB F1 Score:', f1_score(y_test, xgb_preds))
print('XGB AUC:', roc_auc_score(y_test, xgb_proba))

# Cross-validation accuracy for the pipeline on full data
xgb_cv_scores = cross_val_score(xgb_best, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
print('XGB CV Accuracy scores:', xgb_cv_scores)
print('XGB CV Accuracy (mean):', np.mean(xgb_cv_scores))

Fitting 5 folds for each of 243 candidates, totalling 1215 fits
XGB Best Params: {'classifier__colsample_bytree': 0.6, 'classifier__learning_rate': 0.05, 'classifier__max_depth': 5, 'classifier__n_estimators': 500, 'classifier__subsample': 1.0}
XGB Best CV Accuracy: 0.8744628612297702

XGB Confusion Matrix:
[[4636  309]
 [ 510 1058]]

XGB Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.94      0.92      4945
           1       0.77      0.67      0.72      1568

    accuracy                           0.87      6513
   macro avg       0.84      0.81      0.82      6513
weighted avg       0.87      0.87      0.87      6513

XGB Accuracy: 0.874251497005988
XGB Precision: 0.7739575713240673
XGB Recall: 0.6747448979591837
XGB F1 Score: 0.720954003407155
XGB AUC: 0.9317339587503354
XGB CV Accuracy scores: [0.87440504 0.87285012 0.87146806 0.87730344 0.87960688]
XGB CV Accuracy (mean): 0.8751267074620367


### KNN (hyperparameter tuning)

In [30]:
# Hyperparameter tuning for KNN using GridSearchCV
knn_param_grid = {
    'classifier__n_neighbors': [3, 5, 7, 9],
    'classifier__weights': ['uniform', 'distance'],
    'classifier__p': [1, 2]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state_value)

knn_grid = GridSearchCV(knn_model, param_grid=knn_param_grid, cv=cv, scoring='accuracy', n_jobs=-1, verbose=1)

knn_grid.fit(X_train, y_train)
print('KNN Best Params:', knn_grid.best_params_)
print('KNN Best CV Accuracy:', knn_grid.best_score_)

# Evaluate best estimator on test set
knn_best = knn_grid.best_estimator_
knn_preds = knn_best.predict(X_test)
knn_proba = knn_best.predict_proba(X_test)[:, 1]

knn_conf = confusion_matrix(y_test, knn_preds)
print('\nKNN Confusion Matrix:')
print(knn_conf)
print('\nKNN Classification Report:')
print(classification_report(y_test, knn_preds))
print('KNN Accuracy:', np.mean(knn_preds == y_test.values))
print('KNN Precision:', knn_conf[1, 1] / (knn_conf[0, 1] + knn_conf[1, 1]))
print('KNN Recall:', knn_conf[1, 1] / (knn_conf[1, 0] + knn_conf[1, 1]))
print('KNN F1 Score:', f1_score(y_test, knn_preds))
print('KNN AUC:', roc_auc_score(y_test, knn_proba))

# Cross-validation accuracy for the pipeline on full data
knn_cv_scores = cross_val_score(knn_best, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
print('KNN CV Accuracy scores:', knn_cv_scores)
print('KNN CV Accuracy (mean):', np.mean(knn_cv_scores))


Fitting 5 folds for each of 16 candidates, totalling 80 fits


/opt/anaconda3/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


KNN Best Params: {'classifier__n_neighbors': 9, 'classifier__p': 2, 'classifier__weights': 'uniform'}
KNN Best CV Accuracy: 0.8360337655666831

KNN Confusion Matrix:
[[4509  436]
 [ 595  973]]

KNN Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.91      0.90      4945
           1       0.69      0.62      0.65      1568

    accuracy                           0.84      6513
   macro avg       0.79      0.77      0.78      6513
weighted avg       0.84      0.84      0.84      6513

KNN Accuracy: 0.841701212958698
KNN Precision: 0.6905606813342796
KNN Recall: 0.6205357142857143
KNN F1 Score: 0.6536781995297279
KNN AUC: 0.885121605517839
KNN CV Accuracy scores: [0.83878397 0.84136978 0.82939189 0.84628378 0.83891278]
KNN CV Accuracy (mean): 0.8389484402957457
